In [0]:
# Módulos da biblioteca padrão do Python
import json
import logging
import re
import time
import warnings
from datetime import date, datetime, timedelta
from string import Template
from typing import Any, Dict, List, Tuple
from unidecode import unidecode
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.utils import AnalysisException
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    coalesce, col, concat_ws, count, current_date, current_timestamp,
    date_format, date_sub, length, lit, regexp_replace, sha2,
    to_timestamp, trim, udf, upper, when
)
from pyspark.sql.types import (
    DataType, DecimalType, IntegerType, StringType, StructField,
    StructType, TimestampType
)

In [0]:
# Realiza configuracao de logging, storage e token
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logging.getLogger("pyspark").setLevel(logging.WARNING)
logger = logging.getLogger("compass.silver")

storage_account_name = "compassdataprod"
container = "sa-compasslake"
secret_scope_name = "adlsscpkeydata"
secret_key_name   = "adlsstoragekeydata"

try:
    # Recupera o SAS Token do secret scope
    sas_token = dbutils.secrets.get(scope=secret_scope_name, key=secret_key_name)
except Exception as e:
    logger.error(f"Erro ao recuperar SAS Token do secret scope '{secret_scope_name}/{secret_key_name}': {e}")
    raise # Re-levanta a exceção para parar a execução se a credencial for crítica

# Configuração com SAS Token
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account_name}.dfs.core.windows.net", sas_token)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")



In [0]:
# A classe ExecutionMetricsCollector não realiza nenhuma operação pesada no cluster. 
# Como? => Seu único papel é consolidar metadados, como o tempo de execução, e registrar os resultados das ações do Spark que já foram executadas. As operações que demandam maior processamento — como extração, padronização, carga e validação de dados — ocorrem de forma sequencial, disparando seus próprios jobs no cluster. Entre essas operações, a função validate_data() é a que mais impacta a performance, pois envolve múltiplas contagens e filtragens de registros que é executada => após a carga dos dados na tabela Bronze.
class ExecutionMetricsCollector:
    """
    Coleta métricas de execução Spark 
    """

    def __init__(self, spark: SparkSession):
        self.spark = spark
        self.start_time = None
        self.end_time = None
        self.valid_count = 0
        self.invalid_count = 0

    def start_collection(self):
        """Marca início do processo"""
        self.start_time = datetime.now()

    def end_collection(self):
        """Marca fim do processo"""
        self.end_time = datetime.now()

    def set_counts(self, valid_count: int, invalid_count: int):
        """Define as contagens coletadas da execução da validação."""
        self.valid_count = valid_count
        self.invalid_count = invalid_count

    def collect_metrics(
        self,
        validation_results: dict,
        id_app: str,
        layer_lake: str
    ) -> str:
        """
        Gera JSON com métricas da execução.
        """
        if not self.start_time or not self.end_time:
            raise ValueError("start_collection() e end_collection() precisam ser chamados.")

        # Tempo de execução
        total_time = (self.end_time - self.start_time).total_seconds()
        formatted_time = f"{total_time:.2f} s"

        # Contagens (agora obtidas de set_counts)
        count_valid = self.valid_count
        count_invalid = self.invalid_count
        total_records = count_valid + count_invalid
        percentage_valid = (count_valid / total_records * 100) if total_records > 0 else 0.0

        # Monta dicionário de métricas (GENÉRICO)
        metrics = {
            "owner": {
                "dominio": "DOMIMIO_FICT",
                "projeto": "compass",
                "layer_lake": f"{layer_lake}"
            },
            "valid_data": {"count": count_valid, "percentage": percentage_valid},
            "invalid_data": {
                "count": count_invalid,
                "percentage": (count_invalid / total_records * 100) if total_records > 0 else 0.0,
            },
            "total_records": total_records,
            "total_processing_time": formatted_time,
            "validation_results": validation_results,
            "success_count": sum(1 for v in validation_results.values() if isinstance(v, dict) and v.get("status")),
            "error_count": sum(1 for v in validation_results.values() if isinstance(v, dict) and not v.get("status")),
            "_ts": {
                "compass_start_ts": self.start_time.strftime("%Y-%m-%d %H:%M:%S"),
                "compass_end_ts": self.end_time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            "timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "app_id": id_app,
        }

        return json.dumps(metrics, indent=2)

In [0]:
# ===================== Variaveis de entrada via param
date_partition = dbutils.widgets.get("date_partition")
application    = dbutils.widgets.get("application")
layer_source   = dbutils.widgets.get("layer_source")
# date_partition = "2025-09-14"
# application    = "instituicao_reviews"
# layer_source   = "s_compass"


params = {
    "date_partition": date_partition,
    "application":    application,
    "layer_source":   layer_source
}

logging.info("Parâmetros de entrada: %s", json.dumps(params))

# ===================== entrada
data_control = "control_params_compass.data_config"
partition_col = "date_load"

# Instanciar coletor - coleta métricas
collector = ExecutionMetricsCollector(spark)
collector.start_collection()

# ===================== Lê a configuração mais recente da tabela de controle
compass_config = spark.read.table(data_control) \
                           .filter((F.col("source_layer") == layer_source) &
                                   (F.col("table_name_target") == application)) \
                           .orderBy(F.desc("version")) \
                           .take(1)


if not compass_config:
    raise ValueError(f"Nenhuma configuração encontrada para layer {layer_source} e aplicação {application}")



In [0]:
def get_config_compass(cfg: Row) -> tuple:
    """
    Extrai as configurações de uma linha da tabela de controle e retorna-as
    em uma tupla, mantendo a estrutura original.

    Args:
        cfg (Row): A linha da tabela de controle contendo a configuração.

    Returns:
        tuple: Uma tupla com todas as variáveis de configuração em ordem.
    """
    # Converte para dict para facilitar o acesso
    if hasattr(cfg, "asDict"):
        cfg_dict = cfg.asDict()
    else:
        cfg_dict = cfg

    # Extrai os dicionários aninhados, tratando a ausência de chaves
    source_config = cfg_dict.get("source_config", {})
    target_config = cfg_dict.get("target_config", {})
    fallback_config = cfg_dict.get("fallback_config", {})
    rule_control = cfg_dict.get("rule_control", [])

    # Inicializa as regras padrão
    not_empty_rule = "false"
    evolution_mergeschema = "false"
    min_value_rule = None
    max_value_rule = None
    days_back_rule = None

    # Itera sobre as regras de controle
    for rule in rule_control:
        rule_dict = rule.asDict()
        rule_name = rule_dict.get("rule")
        rule_value = rule_dict.get("value")

        if rule_name == "not_empty":
            not_empty_rule = rule_value
        elif rule_name == "evolution_mergeschema":
            evolution_mergeschema = rule_value
        elif rule_name == "min_value":
            min_value_rule = rule_value
        elif rule_name == "max_value":
            max_value_rule = rule_value
        elif rule_name == "days_back":
            days_back_rule = rule_value

    # Atribuicao a variaveis de acordo com a configuração
    table_name_target = cfg_dict.get("table_name_target")
    schema_expected = cfg_dict.get("schema_expected")
    schema_target = cfg_dict.get("schema_target")
    schema_depara = cfg_dict.get("schema_depara")
    version = cfg_dict.get("version")
    last_update = cfg_dict.get("last_modified")

    # Atribuicao das variáveis aninhadas
    source_format = source_config.get("format")
    create_empty_if_missing = fallback_config.get("create_empty_if_missing")
    target_mode = target_config.get("mode")
    target_directory = target_config.get("directory")
    target_format = target_config.get("format")
    partitionBy = target_config.get("partitionBy")

    # Logging estruturado
    logging.info("=== CONFIGURACOES CARREGADAS ===")
    logging.info("Tabela de destino: %s", table_name_target)
    logging.info("Versão: %s", version)
    logging.info("Última modificação: %s", last_update)
    logging.info("Formato de origem: %s", source_format)
    logging.info("Directório de destino: %s", target_directory)
    logging.info("Modo de escrita: %s", target_mode)
    logging.info("Partição por: %s", partitionBy)
    logging.info("Dias retroativos lidos na bronze: %s", days_back_rule)
    logging.info("Regras de validação: not_empty=%s, evolution_mergeschema=%s, min_value=%s, max_value=%s", 
                 not_empty_rule, evolution_mergeschema, min_value_rule, max_value_rule)

    # Retorna todas as variáveis em uma tupla na mesma ordem do seu código original
    return (
        source_format,
        schema_expected,
        create_empty_if_missing,
        table_name_target,
        target_mode,
        target_directory,
        target_format,
        schema_target,
        schema_depara,
        partitionBy,
        not_empty_rule,
        evolution_mergeschema,
        version,
        last_update,
        min_value_rule, max_value_rule, days_back_rule
    )

# ====== CHAMADA EXTRACAO DOS VALORES DE CONFIG ======
if compass_config and len(compass_config) > 0:
    cfg = compass_config[0]

(
    source_format,
    schema_expected,
    create_empty_if_missing,
    table_name_target,
    target_mode,
    target_directory,
    target_format,
    schema_target,
    schema_depara,
    partitionBy,
    not_empty_rule,
    evolution_mergeschema,
    version,
    last_update,
    min_value_rule, max_value_rule, days_back_rule
) = get_config_compass(cfg)


In [0]:
def get_columns(schema_list: List[Row], source_name: str) -> List[str]:
    """
    Filtra uma lista de schemas e retorna os nomes das colunas para uma fonte específica.

    Args:
        schema_list (List[Row]): A lista completa de schemas, onde cada item é um objeto Row.
        source_name (str): O nome da fonte de dados (ex: 'apple_reviews').

    Returns:
        List[str]: Uma lista contendo apenas os nomes das colunas da fonte.
    """
    return [
        row.name_column
        for row in schema_list
        if row.other == source_name
    ]

def func_read_data(
    spark: SparkSession,
    table_name: str,
    columns_to_select: list[str],
    days_load: int
) -> DataFrame:
    """
    Lê dados de uma tabela Delta, seleciona colunas específicas
    e retorna o DataFrame resultante.

    Args:
        spark (SparkSession): Sessão Spark.
        table_name (str): Nome da tabela no catálogo.
        columns_to_select (list[str]): Lista de colunas obrigatórias a selecionar.

    Returns:
        DataFrame: DataFrame com as colunas selecionadas + coluna de auditoria.

    Raises:
        ValueError: Se alguma coluna esperada não existir na tabela.
    """
    try:
        df = spark.read.table(table_name)
        logger.info(f"Tabela '{table_name}' lida com sucesso.")

        current_columns = df.columns
        missing_cols = [c for c in columns_to_select if c not in current_columns]

        if missing_cols:
            raise ValueError(
                f"Tabela '{table_name}' não contém as colunas obrigatórias: {missing_cols}"
            )

        if "date_load" not in current_columns:
            raise ValueError(
                f"Tabela '{table_name}' não contém a coluna obrigatória 'date_load'"
            )

        # Filtro dos últimos (parametro = 365) dias
        df = df.filter(df.date_load >= F.date_sub(F.current_date(), int(days_load)))

        # Adiciona coluna de auditoria
        df = df.withColumn("source_table", lit(table_name))
        selected_cols = columns_to_select + ["source_table"]

        return df.select(selected_cols)

    except AnalysisException as e:
        logger.error(f"Erro ao ler a tabela '{table_name}': {e}")
        raise

In [0]:

columns_apple_reviews = get_columns(schema_expected, 'apple_reviews')
columns_internal_reviews = get_columns(schema_expected, 'internaldb_reviews')

df_apple = func_read_data(
    spark,
    "b_compass.apple_reviews",
    columns_apple_reviews,
    days_back_rule
)

df_internaldb = func_read_data(
    spark,
    "b_compass.internal_db",
    columns_internal_reviews,
    days_back_rule
)


In [0]:
def process_silver(
    df_apple: DataFrame,
    df_internal: DataFrame,
    rules_config: dict
) -> DataFrame:
    """
    Função principal que orquestra as etapas de processamento do pipeline Silver.

    Args:
        df_apple: DataFrame com dados de reviews do Apple Store.
        df_internal: DataFrame com dados de reviews internos.
        rules_config: Dicionário com as regras de filtragem.
    """
    logging.info("Iniciando pipeline Silver de Reviews.")

    # ---------- Unificação e padronização dos dados (Bronze -> Silver)
    df_unified = unify_data(df_apple, df_internal)

    # ---------- Aplicação de regras de qualidade e filtragem
    df_filtered = apply_data_rules(df_unified, rules_config)

    # ---------- Normalização e limpeza de texto
    df_silver = processing_reviews(df_filtered)

    
    return df_silver


def unify_data(df_apple: DataFrame, df_internal: DataFrame) -> DataFrame:
    """
    Unifica e padroniza os DataFrames de reviews externos e internos.
    """
    df_apple_silver = (
        df_apple.withColumn("review_id", sha2("review_id", 256))
        .selectExpr(
            "review_id",
            "author_name as client_id",
            "to_timestamp(updated_at, 'yyyy-MM-dd\\'T\\'HH:mm:ssXXX') as review_date",
            "cast(rating as int) as review_rating",
            "title as review_title",
            "content as review_text",
            "version as review_version",
            "'external' as source_channel",
            "'apple_reviews' as source_system",
            "'NA' as segment",
            "'NA' as service_type",
            "ingestion_ts",
            "date_load",
            "'itunes' as user_agent",
            "upper(app_reference) as app_reference"
        )
    )

    df_internal_silver = (
        df_internal.withColumn("review_id", sha2(concat_ws(":", "client_id", "app_reference"), 256))
        .selectExpr(
            "review_id",
            "client_id",
            "to_timestamp(submission_date) as review_date",
            "feedback_rating as review_rating",
            "'NAO_IDENTIFICADO' as review_title",
            "coalesce(feedback_comment, '') as review_text",
            "cast(source_id as string) as review_version",
            "source_channel",
            "'internaldb_reviews' as source_system",
            "upper(segment) as segment",
            "service_type",
            "ingestion_ts",
            "date_load",
            "user_agent",
            "upper(app_reference) as app_reference"
        )
    )

    return df_apple_silver.unionByName(df_internal_silver, allowMissingColumns=True)

def apply_data_rules(df: DataFrame, rules: dict) -> DataFrame:
    """
    Aplica regras de negócio para filtragem e desduplicação.
    """
    # ---------- Filtro de duplicatas
    df_filtered = df.dropDuplicates(["review_id", "source_system"])

    # ---------- Filtro de valor nulo
    if rules.get("not_empty_rule", "false").lower() == "true":
        df_filtered = df_filtered.filter(col("review_rating").isNotNull())

    # ---------- Filtro por faixa de valor
    min_val = int(rules.get("min_value_rule", "1"))
    max_val = int(rules.get("max_value_rule", "5"))
    df_filtered = df_filtered.filter(
        (col("review_rating") >= min_val) & (col("review_rating") <= max_val)
    )

    return df_filtered


def processing_reviews(df: DataFrame) -> DataFrame:
    """
    Normaliza e limpa campos de texto (título, conteúdo, client_id e service_type),
    substituindo acentos e caracteres indesejados.
    """
    # 1. Função para remover acentos usando Unicode
    def remove_accents(s: str):
        return unidecode(s) if s else None

    remove_accents_udf = udf(remove_accents, StringType())

    df_clean = (
        df.withColumn("source_system", upper(trim("source_system")))
          .withColumn("review_text", upper(trim(remove_accents_udf("review_text"))))
          .withColumn("review_title", upper(trim(remove_accents_udf("review_title"))))
          .withColumn("source_channel", upper(trim(remove_accents_udf("source_channel"))))
          .withColumn("client_id",upper(trim(regexp_replace(remove_accents_udf("client_id"),r"[.\/-]", ""))))
          .withColumn("service_type",upper(trim(regexp_replace(remove_accents_udf("service_type"),r"\s+", "_"))))
          .drop("date_load")
    )

    return df_clean

rules_config = {
    "not_empty_rule": not_empty_rule,
    "min_value_rule": min_value_rule,
    "max_value_rule": max_value_rule
}

df_silver = process_silver(df_apple, df_internaldb, rules_config)
display(df_silver)

In [0]:
def save_data(
        df: DataFrame,
        schema_target: List[Dict[str, str]],
        table_name: str,
        target_mode: str,
        target_format: str,
        partition_value: str
):
    logger.info("Iniciando processo de gravação...")

    partition_ftm = datetime.strptime(partition_value, "%Y-%m-%d").strftime("%Y%m")
    final_df = df.withColumn("date_load", lit(partition_ftm))

    
    if final_df.rdd.isEmpty():  
        logger.warning("Nenhum dado para gravar. Verificar se houve ingestao!")
    else:
        fmt = target_format
        mode = target_mode

        writer = final_df.write.format(fmt).mode(mode).partitionBy("date_load")

        if mode == "overwrite":
            writer = writer.option("replaceWhere", f"date_load = '{partition_ftm}' ")
            logger.info(f"Sobrescrevendo partição {partition_ftm} ")

        writer.saveAsTable(table_name)

    logger.info(f"Dados gravados com sucesso na tabela {table_name}")

save_data(
    df=df_silver,
    schema_target=schema_target,
    table_name=f"{target_directory}.{application}",
    target_mode=target_mode,
    target_format=target_format,
    partition_value=date_partition
)


logger.info("Processo de ingestão finalizado, iniciando o trabalho de coleta de métricas!")

In [0]:
def validate_data(
    spark: SparkSession,
    df: DataFrame,
    compass_config=None,
    ignore_columns: list = None,
    primary_key: list or str = None
) -> tuple:
    """
    Valida um DataFrame de ingestão de forma otimizada, incluindo checagens de valor mínimo e máximo.
    ---
    Args:
        spark (SparkSession): A sessão Spark.
        df (DataFrame): O DataFrame a ser validado.
        compass_config (list): Configurações de validação.
        ignore_columns (list): Lista de colunas a serem ignoradas.
        primary_key (list or str): Colunas que formam a chave primária.
    
    Returns:
        tuple: (valid_records, invalid_records, validation_results, valid_count, invalid_count)
    """
    # 1. Normaliza e prepara configurações
    if ignore_columns is None:
        ignore_columns = []
    
    if isinstance(primary_key, str):
        primary_key = [primary_key]
    elif primary_key is None:
        primary_key = []

    validation_results = {
        "duplicate_check": {"message": None, "status": None, "code": None},
        "null_check": {"message": None, "status": None, "code": None},
        "type_consistency": {"message": None, "status": None, "code": None},
        "total_records": df.count(),
    }

    rule_list = []
    if compass_config:
        silver_rules = compass_config[0].rule_control if hasattr(compass_config[0], 'rule_control') else "[]"
        if isinstance(silver_rules, str):
            try:
                rule_list = json.loads(silver_rules)
            except Exception as e:
                logging.warning(f"Não foi possível interpretar rule_control: {silver_rules} ({e})")
        elif isinstance(silver_rules, (list, tuple)):
            rule_list = [r.asDict() if isinstance(r, T.Row) else r for r in silver_rules]
        else:
            logging.warning(f"Formato inesperado para rule_control: {type(silver_rules)}")

    is_duplicate_col = F.lit(False)
    has_null_issue_col = F.lit(False)
    has_value_range_issue_col = F.lit(False)
    
    # Validação de duplicatas
    if primary_key and any(col in df.columns for col in primary_key):
        pk_cols = [col for col in primary_key if col in df.columns]
        if pk_cols:
            window_spec = Window.partitionBy(*pk_cols).rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
            df = df.withColumn("duplicate_count_check", F.count("*").over(window_spec))
            is_duplicate_col = F.col("duplicate_count_check") > 1

    # Validação de nulos (not_empty)
    null_checks = []
    for rule_dict in rule_list:
        if rule_dict.get("rule") == "not_empty" and str(rule_dict.get("value")).lower() == "true":
            for col_name in df.columns:
                if col_name not in ignore_columns:
                    null_checks.append(F.col(col_name).isNull() | (F.col(col_name) == ''))
    
    if null_checks:
        has_null_issue_col = F.when(F.or_(*null_checks), True).otherwise(False)

    # Nova Lógica: Validação de intervalo de valores (min_value e max_value)
    value_range_checks = []
    has_value_range_rules = False
    min_rules = {r.get("column"): r.get("value") for r in rule_list if r.get("rule") == "min_value"}
    max_rules = {r.get("column"): r.get("value") for r in rule_list if r.get("rule") == "max_value"}

    if min_rules or max_rules:
        has_value_range_rules = True
        for col_name in df.columns:
            if col_name in min_rules:
                min_val = min_rules[col_name]
                value_range_checks.append(F.col(col_name) < min_val)
            if col_name in max_rules:
                max_val = max_rules[col_name]
                value_range_checks.append(F.col(col_name) > max_val)
    
    if value_range_checks:
        has_value_range_issue_col = F.when(F.or_(*value_range_checks), True).otherwise(False)

    # 3. Aplica as validações no DataFrame em uma única passagem
    df_validated = df.withColumn(
        "is_valid",
        ~(is_duplicate_col | has_null_issue_col | has_value_range_issue_col)
    )

    if 'duplicate_count_check' in df_validated.columns:
        df_validated = df_validated.drop('duplicate_count_check')
    
    df_result = df_validated.cache()
    valid_records = df_result.filter(F.col("is_valid"))
    invalid_records = df_result.filter(~F.col("is_valid"))

    valid_count = valid_records.count()
    invalid_count = invalid_records.count()
    
    # 5. Preenche os resultados da validação
    # Duplicatas
    if primary_key and any(col in df.columns for col in primary_key):
        duplicate_count = df_result.filter(is_duplicate_col).count()
        if duplicate_count > 0:
            validation_results["duplicate_check"].update({
                "status": False,
                "code": 409,
                "message": f"Duplicatas encontradas. Total: {duplicate_count} registros."
            })
        else:
            validation_results["duplicate_check"].update({
                "status": True,
                "code": 200,
                "message": "Nenhum registro duplicado encontrado."
            })
    else:
        validation_results["duplicate_check"].update({
            "status": True,
            "code": 200,
            "message": "Nenhuma chave primaria definida para checagem de duplicatas."
        })

    # Nulos
    if null_checks and invalid_count > 0:
        null_issues_df = invalid_records.filter(has_null_issue_col).select(
            *[F.sum(F.when(F.col(col).isNull() | (F.col(col) == ''), 1).otherwise(0)).alias(col) for col in df.columns if col not in ignore_columns]
        )
        null_issues_dict = null_issues_df.collect()[0].asDict()
        null_issues_final = {k: v for k, v in null_issues_dict.items() if v > 0}
        
        if null_issues_final:
            validation_results["null_check"].update({
                "status": False,
                "code": 400,
                "message": f"Valores nulos encontrados: {null_issues_final}"
            })
        else:
            validation_results["null_check"].update({
                "status": True,
                "code": 200,
                "message": "Nenhum valor nulo encontrado (considerando regras aplicadas)."
            })
    else:
        validation_results["null_check"].update({
            "status": True,
            "code": 200,
            "message": "Nenhuma validacao de nulo definida ou nenhum nulo encontrado."
        })
    
    # Validação de intervalo de valores
    if has_value_range_rules:
        value_range_issues_count = df_result.filter(has_value_range_issue_col).count()
        if value_range_issues_count > 0:
            validation_results["type_consistency"].update({
                "status": False,
                "code": 400,
                "message": f"Problemas de intervalo de valores encontrados. Total: {value_range_issues_count} registros."
            })
        else:
            validation_results["type_consistency"].update({
                "status": True,
                "code": 200,
                "message": "Nenhum problema de intervalo de valores encontrado."
            })
    else:
        validation_results["type_consistency"].update({
            "status": True,
            "code": 200,
            "message": "Nenhuma validacao de intervalo de valores definida."
        })
    
    df_result.unpersist()
    
    return valid_records, invalid_records, validation_results, valid_count, invalid_count
    
valid_df, invalid_df, results, valid_count, invalid_count = validate_data(
    spark=spark,
    df=df_silver,
    compass_config=compass_config,
    primary_key=["review_id"]
)

collector.end_collection()

collector.set_counts(valid_count, invalid_count)


# Mapeamento target_table -> camada da arquitetura
LAYER_MAP = {
    "raw": "LANDING",
    "b_compass": "BRONZE",
    "s_compass": "SILVER",
    "g_compass": "GOLD",
    "h_compass": "HARMONIZATION",
}

def map_layer(target_table: str) -> str:
    return LAYER_MAP.get(target_table, f"UNKNOWN_{target_table}".upper())


metrics_json = collector.collect_metrics(
                            validation_results=results,
                            id_app="{}".format(application),
                            layer_lake="{}".format(map_layer(layer_source))
                        )

print(metrics_json)